In [1]:
%load_ext autoreload
%autoreload 2
import torch
from dotenv import load_dotenv
from accelerate import Accelerator
from HuggingFaceModel import HuggingFaceModel
from TrainStrategy import TrainStrategy
from constant import *
from LlmOutputLabelConverter import LlmOutputLabelConverter

/Users/shahidul/dev/project/technical-debt/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/shahidul/dev/project/technical-debt/.venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
load_dotenv()
accelerator = Accelerator()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

Prompts

In [3]:
simple_prompt = PromptTemplate(
    name="definition",
    definition="""
Self-admitted technical debt (SATD) is technical debt admitted by the developer through source code comments. Assign the label of SATD or Not-SATD for each given source code comment.

Here are some examples:\n\n""",
    instruction="Think step by step and assign the label of yes or no for each given test code comment.",
    n_shot_template='### Comment text: """ {{ text }} """',
    n_shot_answer_template="### Label: {{ label }}\n\n",
    line_m_before=0,
    line_n_after=0)

In [ ]:
output_label_converter = LlmOutputLabelConverter({'SATD', 'Not-SATD'}, 'Not-SATD')
prompt_templates = [simple_prompt]

In [ ]:
for prompt_template in prompt_templates:
    for model_name in ['google/flan-t5-small', 'google/flan-t5-base', 'google/flan-t5-large', 'google/flan-t5-xl', 'google/flan-t5-xxl']:
        for shots in [2*n for n in range(6)]:
            t5_model = HuggingFaceModel('detect', model_name, output_label_converter, True)
            t5_model.fit(detect_n_shot_dataset)
            t5_model.predict(detect_test_dataset, DATASET_NAME, prompt_template, TrainStrategy.N_SHOT_TOP, shots, verbose=False)


Dry Run

In [ ]:
for model_name in ['google/flan-t5-small']:
        t5_model = HuggingFaceModel('detect', model_name, output_label_converter,True)
        t5_model.fit(detect_n_shot_dataset)
        t5_model.predict(detect_test_dataset.select(range(10)), DATASET_NAME, simple_prompt, TrainStrategy.N_SHOT_TOP, 2, verbose=False)


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]